# CP6 · Examen blanc de la partie VI — vérification de ta copie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_6/02_examen_notebook.ipynb)

L'examen se fait **sur papier** (`01_examen_sujet.md`, 111 minutes, fiches fermées). Ce notebook ne sert qu'**après** : il vérifie les réponses chiffrées de ta copie. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`, et « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli (⏸️).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_6/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP6`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Après l'examen : vérification automatique

**Ne commence qu'une fois ta copie terminée**, examen rendu. Ce notebook vérifie les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Il ne note pas : les démarches, les justifications, le quiz CP6.Q1, la démonstration CP6.8 et les corrections de CP6.9 b) se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Dans la cellule de chaque question, recopie après chaque lettre **la valeur** écrite sur ta copie, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). Laisse `...` pour une sous-question que tu n'as pas faite. Tu peux aussi coller les lignes de ta copie `04_mes_reponses.md` (« - a) L = … ») : seul ce qui suit le dernier « = » ou le dernier « : » est lu. Efface d'abord ce qui suit ta réponse sur la même ligne (une démarche entre parenthèses, par exemple). Garde les triples guillemets `"""` du début et de la fin : ce que tu écris entre eux est lu comme un texte, et rien de ce que tu tapes là n'arrête la cellule ; « Tout exécuter » (*Run all*) va jusqu'au bilan.
3. Les formes d'écriture, comme sur ta copie : un nombre avec une virgule ou un point (`0,83` ou `0.83`), ou un calcul (`3/8`, `sqrt(7)`, `ln(7)`) ; une liste entre crochets avec des points décimaux, `[0.83, -2.6]`, ou, si tu gardes les virgules décimales, avec des points-virgules entre les éléments, `(0,83 ; -2,6)` (dans une liste, `[4,1, 7,9]` se lit comme quatre nombres) ; une matrice comme la liste de ses lignes, `[[2, -1], [4, 3]]` ; une fraction telle quelle, `3/8` ; des lettres telles quelles, `ABCDE`.
4. Les décimales : quand l'énoncé dit « valeur exacte », donne toutes les décimales de ta valeur (elle n'en a que quelques-unes) ; quand il dit « 4 décimales » (ou 3, ou 2), arrondis à ce nombre, à la fin du calcul seulement. Quand il dit « dans n'importe quel ordre », l'ordre ne compte pas ; ailleurs, l'ordre de ta liste compte.
5. Le bilan, à la fin, dit aussi quelles réponses n'ont pas été vérifiées.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells below

import math
import re
from fractions import Fraction

RESULTS = {}        # sub-ID -> result of wbm.check
_REMINDED = []      # the full reminder is printed once, then a short one
LISTS = {'CP6.10a', 'CP6.10b', 'CP6.10c', 'CP6.10d', 'CP6.10f', 'CP6.10g', 'CP6.1a', 'CP6.1b', 'CP6.1c', 'CP6.1d', 'CP6.1e', 'CP6.1f', 'CP6.2e', 'CP6.3a', 'CP6.3c', 'CP6.4a', 'CP6.4c', 'CP6.5a', 'CP6.6b', 'CP6.6c', 'CP6.6d', 'CP6.7a'}   # the answers that are lists or matrices
TEXTS = {'CP6.5b', 'CP6.5c', 'CP6.5d', 'CP6.6a', 'CP6.9a'}   # the answers read by wbm as texts (fractions, letters)
COLUMNS = {'CP6.6d'}   # the matrices of one column (a flat list is read so)


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans cette cellule, puis exécute-la.")
        _REMINDED.append(True)
        return False
    if "wbm" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


_NAMES = {"sqrt": math.sqrt, "pi": math.pi, "e": math.e, "exp": math.exp, "ln": math.log, "log": math.log,
          "cos": math.cos, "sin": math.sin, "acos": math.acos, "asin": math.asin, "atan": math.atan,
          "degrees": math.degrees, "abs": abs, "math": math, "Fraction": Fraction}
_LINE = re.compile(r"[-*•]?\s*\(?([a-z])\s*[).:]\s*(.*)")   # a lower-case letter: « A : … » is a line of CP6.9
_FRENCH_NUMBER = re.compile(r"[+-]?\d+,\d+")
_COMMAS = ("dans une liste, une virgule sépare deux éléments : « [4,1, 7,9] » se lit comme quatre nombres. Écris "
           "les décimaux avec un point, [4.1, 7.9], ou sépare les éléments par des points-virgules, (4,1 ; 7,9).")
_UNREADABLE = ("je ne lis pas « {} » comme une liste : écris-la entre crochets, avec des points décimaux et des "
               "virgules entre les éléments, comme [0.83, -2.6] (une matrice : la liste de ses lignes, "
               "[[2, -1], [4, 3]]), ou avec des points-virgules entre les éléments, comme (0,83 ; -2,6).")


class Unreadable(ValueError):
    """A value written so that it cannot be read; the message says how to write it."""


def _clean(text):
    """Typographic signs -> Python: − → -, × → *, √2 → sqrt(2), π → pi."""
    for old, new in {"−": "-", "–": "-", "×": "*", "·": "*", "÷": "/", "«": '"', "»": '"', " ": " "}.items():
        text = text.replace(old, new)
    text = re.sub(r"√\s*\(", "sqrt(", text)
    text = re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", text)
    return re.sub(r"(\d)\s*π", r"\1*pi", text).replace("π", "pi").strip()


def _python(text):
    """The value of a number, a list or a calculation, as a calculator reads it (nothing else is allowed)."""
    return eval(text, {"__builtins__": {}}, dict(_NAMES))


def _split(text):
    """The parts of a text separated by semicolons outside brackets."""
    parts, depth, start = [], 0, 0
    for i, char in enumerate(text):
        depth += (char in "[(") - (char in "])")
        if char == ";" and depth == 0:
            parts.append(text[start:i])
            start = i + 1
    return parts + [text[start:]]


def _french_list(text):
    """(0,83 ; -2,6) or [0,83 ; -2,6]: elements separated by semicolons, written with decimal commas."""
    inner = text.strip()
    if inner[:1] in "[(" and inner[-1:] in "])":
        inner = inner[1:-1]
    items = []
    for item in (part.strip() for part in _split(inner)):
        if item[:1] in "[(" and ";" in item:
            items.append(_french_list(item))
        elif _FRENCH_NUMBER.fullmatch(item):
            items.append(float(item.replace(",", ".")))
        else:
            items.append(_python(item))
    return items


def _value_part(text):
    """The value of a line copied from the answer sheet: what follows the last « = » (« L = [[4, 0], [3, 2]] »),
    or the last « : » (« plus petit n : 440 »)."""
    if "=" in text:
        text = text.rsplit("=", 1)[1]
    if ":" in text:
        text = text.rsplit(":", 1)[1]
    return text.strip()


def _given(text):
    return _value_part(text) not in ("", "...", "…")


def read_value(sub_id, text):
    """The value written after a letter, as wbm.check takes it; ... for a sub-question not done."""
    text = _clean(_value_part(text))
    if text in ("", "...", "…"):
        return ...
    if len(text) >= 2 and text[0] == text[-1] and text[0] in "\"'":
        text = text[1:-1].strip()                       # a text between quotes
        if sub_id in TEXTS or sub_id not in LISTS:
            return text
    if sub_id in TEXTS:
        return text                                     # a fraction, letters: wbm reads the text
    if sub_id not in LISTS:
        if _FRENCH_NUMBER.fullmatch(text):
            return text                                 # 0,83: wbm reads a decimal comma in a text
        text = re.sub(r"(\d),(\d)", r"\1.\2", text)     # -ln(0,79): a decimal comma inside a calculation
        try:
            return _python(text)
        except Exception:
            return text                                 # wbm says what it expected
    value = _read_list(text)
    if isinstance(value, (set, frozenset)):
        raise Unreadable("des accolades font un ensemble, qui n'a pas d'ordre : écris une liste entre crochets, dans "
                         "l'ordre demandé par l'énoncé, comme [0.83, -2.6].")
    if sub_id in COLUMNS and isinstance(value, (list, tuple)) and not any(isinstance(x, (list, tuple)) for x in value):
        return [[x] for x in value]                     # a matrix of one column written as the list of its values
    return value


def _read_list(text):
    """A list or a matrix: Python's writing, or semicolons with decimal commas."""
    if ";" in text:
        try:
            return _french_list(text)
        except Exception:
            raise Unreadable(_UNREADABLE.format(text)) from None
    if re.search(r"\d,\d", text) and re.search(r",\s", text):
        raise Unreadable(_COMMAS)                       # [4,1, 7,9]: decimal commas inside a list
    try:
        return _python(text)
    except Exception:
        raise Unreadable(_UNREADABLE.format(text)) from None


def check_question(question, letters, answers):
    """Read the lines « a) value » of ``answers`` and check each value with wbm.check."""
    from wbm.checker import CheckResult

    values, last = {}, None
    for line in answers.split("\n"):              # not splitlines: a pasted \f or \v is no line break
        line = line.split("#")[0].strip()
        if not line:
            continue
        match = _LINE.fullmatch(line)
        if match and match.group(1) in letters:
            last, value = match.group(1), match.group(2)
            if _given(values.get(last, "")) and _given(value) and _value_part(value) != _value_part(values[last]):
                print(f"⚠️ {question} : deux lignes {last}) ; je garde la dernière.")
            if _given(value) or last not in values:
                values[last] = value
        elif last is not None and values[last].count("[") > values[last].count("]"):
            values[last] += " " + line                  # a matrix written on several lines
        elif match:
            print(f"⚠️ {question} : ligne ignorée, « {line} » : ce notebook ne vérifie pas de sous-question "
                  f"{match.group(1)}) ici.")
        else:
            print(f"⚠️ {question} : ligne ignorée, « {line} » (pas de « a) » en tête : une démarche est ignorée, "
                  f"c'est normal ; si c'est une réponse, écris-la comme « a) 0.83 »).")
    for letter in letters:
        sub_id = f"{question}{letter}"
        try:
            value = read_value(sub_id, values.get(letter, "..."))
        except Unreadable as exc:
            print(f"❌ Ex {sub_id} : {exc}")
            RESULTS[sub_id] = CheckResult(sub_id, False, "type", str(exc))
            continue
        RESULTS[sub_id] = wbm.check(sub_id, value)

**CP6.1 — Un vecteur gaussien de dimension 2 : loi marginale, loi conditionnelle, et la loi de μ + Lε**

In [ ]:
# CP6.1: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_1 = r"""
a) ...    # a 2 × 2 matrix, as a list of rows (exact values)
b) ...    # [mean, variance], a list of 2 numbers (exact values)
c) ...    # [mean, variance], a list of 2 numbers (exact values)
d) ...    # [mean, variance], a list of 2 numbers (4 decimals)
e) ...    # a list of 2 numbers (exact values)
f) ...    # a 2 × 2 matrix, as a list of rows (exact values)
"""

if exam_over():
    check_question("CP6.1", "abcdef", answers_CP6_1)

**CP6.2 — La divergence KL entre deux gaussiennes de dimension 2, et une interpolation sphérique**

In [ ]:
# CP6.2: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_2 = r"""
a) ...    # a number (exact value)
b) ...    # a number (exact value)
c) ...    # a number (4 decimals)
d) ...    # an angle in degrees (2 decimals)
e) ...    # a list of 3 numbers (4 decimals)
f) ...    # a number (3 decimals)
"""

if exam_over():
    check_question("CP6.2", "abcdef", answers_CP6_2)

**CP6.3 — Un mélange de deux gaussiennes de variances connues et trois mesures : responsabilités, un pas d'EM sur le poids et les moyennes, et l'écart à l'ELBO d'une affectation dure**

In [ ]:
# CP6.3: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_3 = r"""
a) ...    # a list of 3 numbers (4 decimals)
b) ...    # a number (4 decimals)
c) ...    # [mu1, mu2], a list of 2 numbers (3 decimals)
d) ...    # a number (4 decimals)
"""

if exam_over():
    check_question("CP6.3", "abcd", answers_CP6_3)

**CP6.4 — Une chaîne de Markov à trois états : loi à deux pas, et la valeur d'une politique fixée par le système de Bellman, quand une ligne finit**

In [ ]:
# CP6.4: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_4 = r"""
a) ...    # a list of 3 numbers (exact values)
b) ...    # a number (exact value)
c) ...    # [V(0), V(1), V(2)], a list of 3 numbers (exact values)
d) ...    # a number (exact value)
"""

if exam_over():
    check_question("CP6.4", "abcd", answers_CP6_4)

**CP6.5 — Un jeu à somme nulle 2 × 2 en stratégies mixtes, et W₁ entre deux petits échantillons de même taille**

In [ ]:
# CP6.5: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_5 = r"""
a) ...    # [maximin, minimax], a list of 2 integers
b) ...    # an irreducible fraction, like 3/8
c) ...    # an irreducible fraction, like 3/8
d) ...    # an irreducible fraction, like 3/8
e) ...    # a number (exact value)
f) ...    # a number (4 decimals)
"""

if exam_over():
    check_question("CP6.5", "abcdef", answers_CP6_5)

**CP6.6 — Des distances aux positions : double centrage, matrice de Gram, et une configuration à une isométrie près**

In [ ]:
# CP6.6: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_6 = r"""
a) ...    # an irreducible fraction, like 3/8
b) ...    # a 3 × 3 matrix, as a list of rows (exact values)
c) ...    # a list of 3 numbers, from the largest to the smallest (exact values)
d) ...    # a matrix with 3 rows and 1 column, as a list of rows (exact values)
"""

if exam_over():
    check_question("CP6.6", "abcd", answers_CP6_6)

**CP6.7 — La distance de Fréchet de deux gaussiennes du plan de covariances proportionnelles, et le biais de la distance entre deux moyennes estimées**

In [ ]:
# CP6.7: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_7 = r"""
a) ...    # a list of 2 numbers, from the largest to the smallest (exact values)
b) ...    # a number (exact value)
c) ...    # a number (4 decimals)
d) ...    # a number (exact value)
e) ...    # a number (exact value)
"""

if exam_over():
    check_question("CP6.7", "abcde", answers_CP6_7)

**CP6.9 — Corriger une copie : cinq erreurs, une par chapitre (les lettres seulement)**

In [ ]:
# CP6.9: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_9 = r"""
a) ...    # the letters, in any order, like ABCDE
"""

if exam_over():
    check_question("CP6.9", "a", answers_CP6_9)

**CP6.10 — Problème de synthèse : une file d'attente à trois états suivie de façon gloutonne, puis échantillonnée après la troncature top-p de ses transitions**

In [ ]:
# CP6.10: after each letter, the value written on your answer sheet (... if not done)
answers_CP6_10 = r"""
a) ...    # [f(0), f(1), f(2)], a list of 3 integers
b) ...    # [preperiod, period], a list of 2 integers
c) ...    # a list of integers, by decreasing probability
d) ...    # a list of 3 numbers (4 decimals)
e) ...    # a number (4 decimals)
f) ...    # a 3 × 3 matrix, as a list of rows (4 decimals)
g) ...    # a list of 3 numbers (4 decimals)
"""

if exam_over():
    check_question("CP6.10", "abcdefg", answers_CP6_10)

## Bilan

In [ ]:
# Summary of the automatic checks
import re

EXPECTED = ['CP6.1a', 'CP6.1b', 'CP6.1c', 'CP6.1d', 'CP6.1e', 'CP6.1f', 'CP6.2a', 'CP6.2b', 'CP6.2c', 'CP6.2d', 'CP6.2e', 'CP6.2f', 'CP6.3a', 'CP6.3b', 'CP6.3c', 'CP6.3d', 'CP6.4a', 'CP6.4b', 'CP6.4c', 'CP6.4d', 'CP6.5a', 'CP6.5b', 'CP6.5c', 'CP6.5d', 'CP6.5e', 'CP6.5f', 'CP6.6a', 'CP6.6b', 'CP6.6c', 'CP6.6d', 'CP6.7a', 'CP6.7b', 'CP6.7c', 'CP6.7d', 'CP6.7e', 'CP6.9a', 'CP6.10a', 'CP6.10b', 'CP6.10c', 'CP6.10d', 'CP6.10e', 'CP6.10f', 'CP6.10g']


def natural(sub_id):
    """CP6.2a before CP6.10a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP6\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    missing = [k for k in EXPECTED if k not in RESULTS]
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    if missing:
        print("⚠️ Pas vérifiées (cellule pas exécutée) :", ", ".join(missing))
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; le quiz CP6.Q1 ; la démonstration CP6.8 ; les corrections de CP6.9 b) (seules les lettres de a) sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Remplis ensuite le tableau de ta note, à la fin de ta copie, puis fais la remédiation du corrigé.